# Trajectory Pilot — Experiment 4: Within-stem Split

**上两轮诊断结果**：
- Exp 1 + 2 确认 `ckpt_best.pt (epoch 2) ≈ predict zero`，训练加剧过拟合（pred_std ×6）
- Exp 3 否定了 "event-weighted sampling 是主因"：uniform 版 best_val_mse = 1.5090 vs 原 1.5101

**剩下两个候选根因**：
- D: cross-subject Δσ 漂移（val/test 真实 Δσ 比 train 大 50%+）
- E: 未来 10s ranktrace 本身在 cross-subject 下不可学

**本实验**：把 split 换成 within-stem 时间切分（每个 stem 前 70% = train，中 15% = val，后 15% = test，每段之间有 10s gap 防止标签泄漏）。

**判读规则**：

| `best_val_ccc` | `best_epoch` | `train_ccc` | 结论 |
|---|---|---|---|
| > 0.2 | > 3 | > 0.3 | 任务在 within-stem 可学 → cross-subject 是唯一硬瓶颈 |
| 0.05–0.2 | > 3 | > 0.3 | 部分信号，但模型容量/正则需要调 |
| ≈ 0 | 2 | > 0.3 | 架构/任务本身有问题（train 能 fit，val 不行，即使同 session）|
| 所有都 ≈ 0 | — | ≈ 0 | 数据或训练 pipeline 有 bug（之前没发现） |

**前置**：需要 `splits/within_subject.json`（已确认存在，87 stems）。

In [1]:
# Cell 1: Mount + cd + GPU check
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/ProjectExperiment
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

Mounted at /content/drive
/content/drive/MyDrive/ProjectExperiment
NVIDIA L4, 23034 MiB


---
## 预检：确认 within-stem 切分行为正确

跑训练前先验证 anchor 切分逻辑，防止白跑 10 分钟才发现 bug。

In [ ]:
# Cell 2: Verify within-stem split produces disjoint output ranges per stem
import sys, yaml
if '/content/drive/MyDrive/ProjectExperiment' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive/ProjectExperiment')

import src.data.datamodules  # triggers registration
from src.core.registry import DATAMODULES

data_cfg = {
    'name': 'amucs_trajectory',
    'data_root': '/content/drive/MyDrive/AmuCS_experiment/features/aligned',
    'labels_seq_path': '/content/drive/MyDrive/AmuCS_experiment/labels/labels_arousal_seq.json',
    'events_path': '/content/drive/MyDrive/AmuCS_experiment/labels/events_index.json',
    'split_path': '/content/drive/MyDrive/AmuCS_experiment/splits/within_subject.json',
    'modalities': ['video'],
    'pre_s': 20.0, 'post_s': 10.0, 'dt': 0.2,
    'train_stride_s': 2.0, 'val_stride_s': 2.0, 'test_stride_s': 1.0,
    'normalize': True,
    'batch_size': 256,
    'num_workers': 4,
    'within_stem_split': True,
    'within_stem_ratios': [0.7, 0.15, 0.15],
    'event_weighted': False,  # speed up preflight
    'modality_dir_map': {'video': 'video_clip', 'km': 'km', 'telem': 'telem'},
}

dm = DATAMODULES.build('amucs_trajectory', data_cfg)

print(f'\n=== Anchor counts per split ===')
print(f'  train: {len(dm._train_ds._anchors)}')
print(f'  val:   {len(dm._val_ds._anchors)}')
print(f'  test:  {len(dm._test_ds._anchors) if dm._test_ds else 0}')
print(f'  train Δσ: {dm.delta_sigma:.3f}')

# Verify disjoint output ranges for a sample stem
print(f'\n=== Sample stem verification ===')
sample_stem = dm._train_ds.stems[0]
def stem_output_range(ds, stem):
    idxs = [i for (s, i) in ds._anchors if s == stem]
    if not idxs:
        return None
    return (min(idxs), max(idxs) + ds.post_steps)

tr_r = stem_output_range(dm._train_ds, sample_stem)
va_r = stem_output_range(dm._val_ds, sample_stem)
te_r = stem_output_range(dm._test_ds, sample_stem)
print(f'  stem={sample_stem}  train out: {tr_r}  val out: {va_r}  test out: {te_r}')
if tr_r and va_r:
    assert tr_r[1] <= va_r[0], f'LEAK: train output end {tr_r[1]} > val output start {va_r[0]}'
if va_r and te_r:
    assert va_r[1] <= te_r[0], f'LEAK: val output end {va_r[1]} > test output start {te_r[0]}'
print('  ✅ output ranges disjoint — no label leak')

---
## 跑 within-stem 训练（~10 min, single_video / seed 0）

In [ ]:
# Cell 3: Run the within-stem sweep
# 用 cd && python 保证 cwd 正确，不依赖 Cell 1 的 %cd 是否跑过
!cd /content/drive/MyDrive/ProjectExperiment && python -u scripts/run_experiment.py \
    --sweep configs/sweeps/trajectory_pilot_within_stem.yaml \
    --workers 1 \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot

---
## 三方对比：cross-subject（原）vs uniform vs within-stem

In [ ]:
# Cell 4: Aggregate metrics.json from all 3 runs and print comparison
import glob, json
from pathlib import Path

def latest_run(base_glob):
    matches = sorted(glob.glob(base_glob))
    valid = [m for m in matches if (Path(m) / 'metrics.json').exists()]
    assert valid, f'no completed run matching {base_glob}'
    return valid[-1]

BASE = '/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot'
runs = {
    'cross-subject (event-weighted, original)':
        latest_run(f'{BASE}/cross_subject/eft_trajectory_pilot_3seed/single_video/*seed0*'),
    'cross-subject (uniform, Exp 3)':
        latest_run(f'{BASE}/cross_subject/eft_trajectory_pilot_uniform_3seed/single_video/*seed0*'),
    'within-stem (event-weighted, Exp 4)':
        latest_run(f'{BASE}/within_subject/eft_trajectory_pilot_within_stem_3seed/single_video/*seed0*'),
}

def fmt(v, w=10, d=4):
    if v is None or (isinstance(v, float) and v != v): return f'{"?":>{w}}'
    if isinstance(v, float): return f'{v:>{w}.{d}f}'
    return f'{str(v):>{w}}'

rows = []
for label, rd in runs.items():
    m = json.load(open(Path(rd) / 'metrics.json'))
    rows.append({
        'label': label,
        'best_ep': m.get('best_epoch'),
        'best_val_mse': m.get('best_val_metric'),
        'total_ep': m.get('total_epochs'),
        'train_ccc_last': m.get('train_ccc'),
        'val_ccc_last': m.get('val_ccc'),
        'test_mse': m.get('test_mse'),
        'test_ccc': m.get('test_ccc'),
    })

print(f"{'run':<45} {'best_ep':>8} {'best_val_mse':>13} {'total_ep':>9} {'tr_ccc':>8} {'va_ccc':>8} {'te_mse':>8} {'te_ccc':>8}")
print('-' * 115)
for r in rows:
    print(f"{r['label']:<45} {fmt(r['best_ep'], 8, 0)} {fmt(r['best_val_mse'], 13, 4)} {fmt(r['total_ep'], 9, 0)} "
          f"{fmt(r['train_ccc_last'], 8, 3)} {fmt(r['val_ccc_last'], 8, 3)} {fmt(r['test_mse'], 8, 3)} {fmt(r['test_ccc'], 8, 3)}")

In [ ]:
# Cell 5: Verdict
within = rows[2]
orig   = rows[0]

best_ep = within['best_ep']
val_ccc = within['val_ccc_last']
tr_ccc  = within['train_ccc_last']
best_val_mse = within['best_val_mse']

print(f"within-stem:  best_ep={best_ep}  best_val_mse={best_val_mse:.4f}  train_ccc={tr_ccc:.3f}  val_ccc={val_ccc:.3f}\n")

if val_ccc is None or tr_ccc is None:
    print('❌ metrics 缺失，无法判定')
elif val_ccc > 0.2 and best_ep > 3:
    print('✅ 根因 E 排除：within-stem 任务可学，val_ccc > 0.2')
    print('   → cross-subject 是唯一硬瓶颈')
    print('   下一步：per-subject 归一化 / subject-adversarial / domain adaptation')
elif val_ccc > 0.05 and best_ep > 3:
    print('🟡 部分信号：within-stem val_ccc 有起色但不够强')
    print('   → 模型容量或正则可以调；cross-subject 仍是主因')
elif best_ep <= 3 and tr_ccc > 0.3 and val_ccc < 0.1:
    print('❌ 根因 E 高度怀疑：train 能 fit (train_ccc > 0.3) 但 val 仍 ≈ 0')
    print('   说明即使消除 cross-subject 影响，模型学到的东西依然不迁移')
    print('   → 架构瓶颈嫌疑重新升级 / 任务本身预测力不足')
    print('   下一步：简化任务（单点 Δ 而非 51 点）或换 seq2seq head 对比')
elif tr_ccc < 0.1:
    print('❌ train_ccc 都很低，可能是数据/pipeline bug')
    print('   下一步：查学习率、梯度、特征加载路径')
else:
    print('🟡 ambiguous — 人工判读')

---
## 根据 Verdict 的下一步决策树

```
within-stem val_ccc > 0.2
├── YES → cross-subject 是瓶颈
│         做法: per-subject Δσ 归一化 / 引入 subject embedding / adversarial
│
└── NO  → 看 train_ccc
          ├── train_ccc > 0.3: 架构/任务有问题
          │     做法 A: 简化 target 到单点 Δ@+5s，看能否学
          │     做法 B: 换 seq2seq decoder head（把 pooled 瓶颈拆掉）
          │     做法 C: 缩短 post_s 到 3-5 秒
          │
          └── train_ccc < 0.1: pipeline bug
                做法: 查 LR / 梯度 / feature 加载是否对齐
```

---
## Exp 4b: Within-stem zero-prediction baseline

Cell 5 的 verdict 只基于 `val_ccc/train_ccc`。还需要知道 **within-stem 的零预测 MSE**，才能判断 `best_val_mse = 1.474` 是"略胜 zero"还是"仍然输给 zero"。

- 若 model < zero × 0.95 → cross-subject 是唯一瓶颈
- 若 model ≈ zero (±2%) → 架构也有嫌疑
- 若 model > zero × 1.02 → 架构/任务本质问题

复用 Cell 2 的 `dm`（如果还在内存），否则重建（~30s）。

In [ ]:
# Cell 6: Within-stem zero baseline + final verdict
import sys
if '/content/drive/MyDrive/ProjectExperiment' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive/ProjectExperiment')

# Try reusing dm from preflight Cell 2; rebuild if not in scope or wrong config
try:
    assert getattr(dm, 'within_stem_split', False), 'existing dm is not within-stem'
    print('Reusing dm from preflight cell')
except (NameError, AssertionError):
    print('Rebuilding datamodule (~30s)...')
    import src.data.datamodules  # triggers registration
    from src.core.registry import DATAMODULES
    data_cfg = {
        'name': 'amucs_trajectory',
        'data_root': '/content/drive/MyDrive/AmuCS_experiment/features/aligned',
        'labels_seq_path': '/content/drive/MyDrive/AmuCS_experiment/labels/labels_arousal_seq.json',
        'events_path': '/content/drive/MyDrive/AmuCS_experiment/labels/events_index.json',
        'split_path': '/content/drive/MyDrive/AmuCS_experiment/splits/within_subject.json',
        'modalities': ['video'],
        'pre_s': 20.0, 'post_s': 10.0, 'dt': 0.2,
        'train_stride_s': 2.0, 'val_stride_s': 2.0, 'test_stride_s': 1.0,
        'normalize': True, 'batch_size': 256, 'num_workers': 4,
        'within_stem_split': True,
        'within_stem_ratios': [0.7, 0.15, 0.15],
        'event_weighted': False,
        'modality_dir_map': {'video': 'video_clip', 'km': 'km', 'telem': 'telem'},
    }
    dm = DATAMODULES.build('amucs_trajectory', data_cfg)

def zero_mse(loader):
    if loader is None:
        return float('nan')
    tot_sq, n = 0.0, 0
    for b in loader:
        y = b['y'].float()
        tot_sq += (y ** 2).sum().item()
        n += y.numel()
    return tot_sq / max(n, 1)

print('\nComputing zero baseline on val + test ...', flush=True)
val_zero  = zero_mse(dm.val_dataloader())
test_zero = zero_mse(dm.test_dataloader())

print(f'\n=== within-stem zero-prediction MSE ===')
print(f'  train Δσ (normalizer): {dm.delta_sigma:.4f}')
print(f'  val  zero_mse = {val_zero:.4f}')
print(f'  test zero_mse = {test_zero:.4f}')

# Compare to within-stem model's best
import json, glob
from pathlib import Path
BASE = '/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/within_subject/eft_trajectory_pilot_within_stem_3seed/single_video'
run = sorted(glob.glob(f'{BASE}/*seed0*'))[-1]
m = json.load(open(Path(run) / 'metrics.json'))
best_val  = m['best_val_metric']
test_mse  = m.get('test_mse')
val_mse_f = m.get('val_mse')

print(f'\n=== within-stem: model vs zero baseline ===')
print(f'  val:  best_val_mse={best_val:.4f}  final_val_mse={val_mse_f:.4f}  zero={val_zero:.4f}  ratio(best/zero)={best_val/val_zero:.3f}')
print(f'  test: test_mse={test_mse:.4f}  zero={test_zero:.4f}  ratio={test_mse/test_zero:.3f}')

# Final verdict
print(f'\n=== Final verdict ===')
if best_val < val_zero * 0.95:
    print('✅ within-stem model BEATS zero baseline by > 5%')
    print('   → cross-subject IS the main bottleneck')
    print('   → 下一步: per-subject Δσ 归一化 / subject embedding / 正式 3 模态 × 3 seed 跑 within-stem')
elif best_val < val_zero * 1.02:
    print('🟡 within-stem model ≈ zero baseline (within ±2%)')
    print('   → 微弱信号。cross-subject 不是唯一瓶颈')
    print('   → 架构嫌疑升级：换 seq2seq head / 简化 target（单点 Δ@+5s）/ 缩短 post_s 到 3s')
else:
    print('❌ within-stem model WORSE than zero baseline')
    print('   → 架构或任务本身有更深问题，cross-subject 是次要因素')
    print('   → 下一步: seq2seq decoder head 必做，或重新考虑目标定义')

---
# Exp 5 + Exp 6 — 排查瓶颈是"距离"还是"51 维并排输出"

Exp 4 结论：within-stem 模型只比零预测好 0.6%（val）、还输 5.5%（test）。说明 cross-subject 不是主瓶颈，**架构/任务本质有问题**。

再做两个小实验定位：

| 实验 | 变动 | 预期的区分力 |
|---|---|---|
| **Exp 5** 缩短 horizon | `post_s: 10 → 3`，`out_dim: 51 → 16` | 如 val_ccc > 0.2，说明 10s 太远 |
| **Exp 6** 单点 target | 预测 `Δ@+5s` 一个标量，`out_dim: 1` | 如单点也学不会，是模型读不懂输入；如单点可学而 51 点不行，pooled→51 维瓶颈属实 |

两个实验都用 within-stem split（最有利条件），各 ~10 min。

In [ ]:
# Cell 7: Exp 5 — shorter horizon (post_s=3, out_dim=16)
!cd /content/drive/MyDrive/ProjectExperiment && python -u scripts/run_experiment.py \
    --sweep configs/sweeps/trajectory_pilot_within_short.yaml \
    --workers 1 \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot

In [ ]:
# Cell 8: Exp 6 — single-point target (Δ@+5s, out_dim=1)
!cd /content/drive/MyDrive/ProjectExperiment && python -u scripts/run_experiment.py \
    --sweep configs/sweeps/trajectory_pilot_within_single.yaml \
    --workers 1 \
    --data_root /content/drive/MyDrive/AmuCS_experiment/features/aligned \
    --labels_root /content/drive/MyDrive/AmuCS_experiment/labels \
    --splits_root /content/drive/MyDrive/AmuCS_experiment/splits \
    --runs_root /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot

In [ ]:
# Cell 9: Exp 5 + 6 analysis — zero baselines per config + verdict
import sys, glob, json
from pathlib import Path
import numpy as np

if '/content/drive/MyDrive/ProjectExperiment' not in sys.path:
    sys.path.insert(0, '/content/drive/MyDrive/ProjectExperiment')

# Compute zero baseline from labels directly (matches datamodule exactly — verified in Exp 4)
LABELS = '/content/drive/MyDrive/AmuCS_experiment/labels/labels_arousal_seq.json'
SPLIT  = '/content/drive/MyDrive/AmuCS_experiment/splits/within_subject.json'
with open(LABELS, encoding='utf-8-sig') as f: labels = json.load(f)
with open(SPLIT,  encoding='utf-8-sig') as f: split = json.load(f)
stems = sorted(set(split['train']) & set(labels.keys()))

DT = 0.2
PRE_STEPS = 100
R_TR, R_VA = 0.7, 0.15

def compute_zero_and_ref(post_s, target_mode, target_offset_s):
    post_steps = int(round(post_s / DT)) + 1
    k_offset = int(round(target_offset_s / DT))
    stride_tr, stride_va, stride_te = 10, 10, 5

    def ranges(L):
        i_min = PRE_STEPS
        i_max = L - post_steps
        if i_max <= i_min: return None
        span = i_max - i_min
        tr_end = i_min + int(span * R_TR)
        va_start = tr_end + post_steps
        va_end = va_start + int(span * R_VA)
        te_start = va_end + post_steps
        return {'train': (i_min, tr_end, stride_tr),
                'val':   (va_start, va_end, stride_va),
                'test':  (te_start, i_max, stride_te)}

    # delta_sigma from train anchors
    deltas_tr = []
    for stem in stems:
        y = np.asarray(labels[stem]['values'], dtype=np.float64)
        r = ranges(len(y))
        if r is None: continue
        lo, hi, st = r['train']
        for i in range(lo, hi + 1, st):
            if target_mode == 'single_point':
                deltas_tr.append(np.array([y[i + k_offset] - y[i]]))
            else:
                deltas_tr.append(y[i:i + post_steps] - y[i])
    sigma = float(np.std(np.concatenate(deltas_tr))) or 1.0

    # zero_mse per split
    def zero(split_name):
        sq, n = 0.0, 0
        for stem in stems:
            y = np.asarray(labels[stem]['values'], dtype=np.float64)
            r = ranges(len(y))
            if r is None: continue
            lo, hi, st = r[split_name]
            if hi <= lo: continue
            for i in range(lo, hi + 1, st):
                if target_mode == 'single_point':
                    d = (y[i + k_offset] - y[i]) / sigma
                    sq += d * d
                    n += 1
                else:
                    d = (y[i:i + post_steps] - y[i]) / sigma
                    sq += float(np.sum(d * d))
                    n += post_steps
        return sq / max(n, 1)

    return sigma, zero('val'), zero('test')

# Exp 4 (trajectory, post_s=10) — reuse known numbers
s4, v4, t4 = compute_zero_and_ref(10.0, 'trajectory', 5.0)
# Exp 5 (trajectory, post_s=3)
s5, v5, t5 = compute_zero_and_ref(3.0,  'trajectory', 5.0)
# Exp 6 (single_point @ +5s, post_s=10 stays as anchor filter)
s6, v6, t6 = compute_zero_and_ref(10.0, 'single_point', 5.0)

# Load model best_val_mse from each run
BASE = '/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/within_subject'
def read_best(glob_path):
    matches = sorted(glob.glob(glob_path))
    valid = [m for m in matches if (Path(m) / 'metrics.json').exists()]
    if not valid: return None
    m = json.load(open(Path(valid[-1]) / 'metrics.json'))
    return m

r4 = read_best(f'{BASE}/eft_trajectory_pilot_within_stem_3seed/single_video/*seed0*')
r5 = read_best(f'{BASE}/eft_trajectory_pilot_within_short_3seed/single_video/*seed0*')
r6 = read_best(f'{BASE}/eft_trajectory_pilot_within_single_3seed/single_video/*seed0*')

print(f"{'exp':<35} {'best_ep':>8} {'best_val_mse':>13} {'zero_val':>10} {'ratio':>8} {'tr_ccc':>8} {'va_ccc':>8} {'te_ccc':>8}")
print('-' * 110)
for name, r, zv, sig in [
    ('Exp 4: trajectory, post_s=10',  r4, v4, s4),
    ('Exp 5: trajectory, post_s=3',   r5, v5, s5),
    ('Exp 6: single-point @+5s',      r6, v6, s6),
]:
    if r is None:
        print(f'{name:<35}   (no run found — skip)')
        continue
    bvm = r['best_val_metric']
    tr = r.get('train_ccc')
    va = r.get('val_ccc')
    te = r.get('test_ccc')
    ratio = bvm / zv
    def fmt(v, w=8, d=4):
        if v is None or (isinstance(v, float) and v != v): return f'{"?":>{w}}'
        return f'{v:>{w}.{d}f}' if isinstance(v, float) else f'{str(v):>{w}}'
    print(f"{name:<35} {fmt(r.get('best_epoch'), 8, 0)} {fmt(bvm, 13)} {fmt(zv, 10)} {fmt(ratio, 8, 3)} {fmt(tr, 8, 3)} {fmt(va, 8, 3)} {fmt(te, 8, 3)}")

print('\n=== Verdict ===')
checks = []
for label, r, zv in [('Exp 5', r5, v5), ('Exp 6', r6, v6)]:
    if r is None:
        checks.append((label, 'skip', None))
        continue
    bvm = r['best_val_metric']
    va  = r.get('val_ccc') or 0.0
    if bvm < zv * 0.90 or va > 0.15:
        checks.append((label, 'BEATS zero clearly', bvm / zv))
    elif bvm < zv * 0.98:
        checks.append((label, 'slight improvement', bvm / zv))
    else:
        checks.append((label, 'flat (≈ zero)', bvm / zv))

for label, status, ratio in checks:
    r_str = f' ratio={ratio:.3f}' if ratio is not None else ''
    print(f'  {label}: {status}{r_str}')

# Decision tree
exp5_ok = checks[0][1] in ('BEATS zero clearly', 'slight improvement') if checks[0][1] != 'skip' else None
exp6_ok = checks[1][1] in ('BEATS zero clearly', 'slight improvement') if checks[1][1] != 'skip' else None

print()
if exp5_ok and exp6_ok:
    print('→ Horizon 和 single-point 都有帮助：长轨迹的 51 维并排输出 + 10s 距离都是问题')
    print('  下一步：seq2seq head + 较短 horizon 联合尝试')
elif exp5_ok and not exp6_ok:
    print('→ 缩短 horizon 有效但单点不行。反直觉 —— 可能是单点信噪比太差。建议重看 train/val_ccc')
elif not exp5_ok and exp6_ok:
    print('→ 单点能学但 51 点轨迹不行 — pooled→51 维 MLP 瓶颈属实')
    print('  下一步：换 seq2seq decoder head（新 head，让每个输出点独立从 encoder tokens 读）')
elif exp5_ok is False and exp6_ok is False:
    print('❌ 两个简化都不能突破零预测 → past 20s 对未来（任何尺度）基本没预测力')
    print('  下一步：重新考虑范式。选项：')
    print('    - 换成分类任务（上升/持平/下降）')
    print('    - 改用纯过去窗口重建（self-supervised）')
    print('    - 回到原 3-class state+trend 任务')
else:
    print('🟡 某实验没跑成功，人工判读')

---
# Exp 7 — 事件锁相分层评估（within-subject Exp 4 + Exp 5）

之前只有训练期 uniform-stride CCC（overall）。这里对已有 checkpoint 做分层评估：
- `event` 分层：anchor ±10s 内有任何 event（bomb/kill/round）
- `quiet` 分层：anchor 远离所有 event
- 每层单独算 CCC、event_corr_mean、pred_std/gt_std、peak_f1

**决策逻辑**：
- event CCC >> overall CCC → 信号集中在事件区，uniform eval 稀释了它 → 后续用 event CCC 做主 metric
- event ≈ quiet 且 pred_std/gt_std < 0.1 → 又是 predict-zero 陷阱
- event ≈ quiet 且 pred_std/gt_std > 0.2 → 模型在动但没对准信号

Exp 6（single_point，post_steps=1）先跳过 —— Pearson/peak 类 metric 需要 ≥2 点。


In [ ]:
# Cell 10: 对 Exp 4 + Exp 5 跑事件锁相分层评估（best + last）
# 输出：每个 run_dir 下会产生 stratified_eval.json (from ckpt_best) 和 stratified_eval_ckpt_last.json

!cd /content/drive/MyDrive/ProjectExperiment && \
for RUN in \
  "/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/within_subject/eft_trajectory_pilot_within_stem_3seed/single_video/2026-04-24_10-26-32__amucs_trajectory__eft__video__seed0" \
  "/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/within_subject/eft_trajectory_pilot_within_short_3seed/single_video/2026-04-24_11-52-36__amucs_trajectory__eft__video__seed0"; do \
  echo "========== $RUN ==========" ; \
  python scripts/evaluate_trajectory.py --run_dir "$RUN" --splits val test --ckpt_name ckpt_best.pt ; \
  python scripts/evaluate_trajectory.py --run_dir "$RUN" --splits val test --ckpt_name ckpt_last.pt ; \
done


In [ ]:
# Cell 11: 聚合 Exp 4 + Exp 5 的 stratified_eval JSON，做决策对比表
import json
from pathlib import Path

RUNS = {
    "Exp4_post10": "/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/within_subject/eft_trajectory_pilot_within_stem_3seed/single_video/2026-04-24_10-26-32__amucs_trajectory__eft__video__seed0",
    "Exp5_post3":  "/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/within_subject/eft_trajectory_pilot_within_short_3seed/single_video/2026-04-24_11-52-36__amucs_trajectory__eft__video__seed0",
}
CKPT_FILES = {"best": "stratified_eval.json", "last": "stratified_eval_ckpt_last.json"}

rows = []
for exp_name, rd in RUNS.items():
    rd = Path(rd)
    for ckpt_label, fname in CKPT_FILES.items():
        p = rd / fname
        if not p.exists():
            print(f"[miss] {exp_name} / {ckpt_label}: {p}")
            continue
        with p.open("r", encoding="utf-8") as f:
            data = json.load(f)
        for split in ("val", "test"):
            strata = data["results"].get(split, {})
            for stratum in ("event", "quiet", "overall"):
                m = strata.get(stratum, {})
                if not m or m.get("n", 0) == 0:
                    continue
                rows.append({
                    "exp": exp_name,
                    "ckpt": ckpt_label,
                    "split": split,
                    "stratum": stratum,
                    "n": m.get("n"),
                    "mse": m.get("mse"),
                    "ccc": m.get("ccc"),
                    "event_corr_mean": m.get("event_corr_mean"),
                    "pred_std": m.get("pred_std"),
                    "gt_std": m.get("gt_std"),
                    "pred_std_over_gt_std": m.get("pred_std_over_gt_std"),
                    "peak_f1": m.get("peak_f1"),
                })

# 打印对比表
import pandas as pd
df = pd.DataFrame(rows)
if len(df) == 0:
    print("没有找到任何 stratified_eval JSON — 先跑 Cell 10")
else:
    pd.set_option("display.float_format", lambda v: f"{v:.4f}" if v is not None and isinstance(v, float) else str(v))
    print("\n===== 完整对比表 =====")
    print(df.to_string(index=False))

    # 决策关键视图：event vs quiet 的 CCC 差距 + pred_std/gt_std
    print("\n===== 决策视图（test split）=====")
    test_df = df[df["split"] == "test"].copy()
    pivot_ccc = test_df.pivot_table(index=["exp","ckpt"], columns="stratum", values="ccc")
    pivot_psr = test_df.pivot_table(index=["exp","ckpt"], columns="stratum", values="pred_std_over_gt_std")
    print("\n-- CCC per stratum --")
    print(pivot_ccc)
    print("\n-- pred_std/gt_std per stratum --")
    print(pivot_psr)

    # 决策判断
    print("\n===== 判断 =====")
    for (exp, ckpt), row_ccc in pivot_ccc.iterrows():
        ov = row_ccc.get("overall")
        ev = row_ccc.get("event")
        qt = row_ccc.get("quiet")
        psr_ov = pivot_psr.loc[(exp, ckpt)].get("overall") if (exp, ckpt) in pivot_psr.index else None
        verdict = []
        if psr_ov is not None and psr_ov < 0.1:
            verdict.append(f"predict-zero (pred_std/gt_std={psr_ov:.3f})")
        if ev is not None and ov is not None and ev > 2 * max(abs(ov), 0.01):
            verdict.append(f"事件区信号集中 (event={ev:.3f} vs overall={ov:.3f}) ← 考虑 A 方向")
        if ev is not None and qt is not None and abs(ev - qt) < 0.02:
            verdict.append(f"事件 vs 静音 差别 < 0.02 (ev={ev:.3f}, qt={qt:.3f})")
        print(f"{exp} / {ckpt}: {'; '.join(verdict) if verdict else 'OK'}")


---
# Exp 8 — Exp 6 (single_point) 事件锁相评估

**目的**：决定 A vs B 方向的最后一块证据。

对照：
- Exp 4 (post_s=10, out_dim=51) test overall CCC = 0.046
- Exp 6 (post_s=10, out_dim=1) metrics.json test_ccc = 0.087 ← **同 horizon 下 2x**

如果 Exp 6 的 event vs quiet CCC 比值 > 2x → output-dim 瓶颈确认 → 走 B (seq2seq)。
如果 event vs quiet < 1.5x（和 Exp 4/5 一样） → 单点预测也信号分散 → 走 A (正式 sweep)。

event_locked.py 对 post_steps=1 已有防御（NaN 而非崩溃），不需要 patch。
peak_f1 / event_corr_mean 会是 NaN，但 CCC / MSE / pred_std/gt_std 正常。


In [2]:
# Cell 12: 对 Exp 6 (single_point, out_dim=1) 跑事件锁相分层评估

!cd /content/drive/MyDrive/ProjectExperiment && \
RUN="/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/within_subject/eft_trajectory_pilot_within_single_3seed/single_video/2026-04-24_11-59-33__amucs_trajectory__eft__video__seed0" && \
python scripts/evaluate_trajectory.py --run_dir "$RUN" --splits val test --ckpt_name ckpt_best.pt && \
python scripts/evaluate_trajectory.py --run_dir "$RUN" --splits val test --ckpt_name ckpt_last.pt


[load] config from /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/within_subject/eft_trajectory_pilot_within_single_3seed/single_video/2026-04-24_11-59-33__amucs_trajectory__eft__video__seed0/config.yaml
[load] checkpoint from /content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/within_subject/eft_trajectory_pilot_within_single_3seed/single_video/2026-04-24_11-59-33__amucs_trajectory__eft__video__seed0/ckpt_best.pt
[AMuCSTrajectoryDataModule] train-set Δσ = 19.4047
[AMuCSTrajectoryDataModule] train sampler = event-weighted (tau_s=10.0)
[val] collecting predictions ...
[val] 4855 anchors  pred_shape=(4855, 1)  target_shape=(4855, 1)
[val] strata: event=4332 (89.2%) quiet=523 (10.8%)
[test] collecting predictions ...
[test] 7920 anchors  pred_shape=(7920, 1)  target_shape=(7920, 1)
[test] strata: event=4101 (51.8%) quiet=3819 (48.2%)

=== VAL ===
stratum        n      mse     ccc pred_std  gt_std  peak_f1  lead_s  amp_err  ev_corr
---------------------------------

In [3]:
# Cell 13: 聚合 Exp 4 + 5 + 6 的 stratified_eval JSON，完整决策视图
import json
from pathlib import Path
import pandas as pd

BASE = Path("/content/drive/MyDrive/AmuCS_experiment/runs/trajectory_pilot/within_subject")
RUNS = {
    "Exp4_post10_d51": BASE / "eft_trajectory_pilot_within_stem_3seed/single_video/2026-04-24_10-26-32__amucs_trajectory__eft__video__seed0",
    "Exp5_post3_d16":  BASE / "eft_trajectory_pilot_within_short_3seed/single_video/2026-04-24_11-52-36__amucs_trajectory__eft__video__seed0",
    "Exp6_post10_d1":  BASE / "eft_trajectory_pilot_within_single_3seed/single_video/2026-04-24_11-59-33__amucs_trajectory__eft__video__seed0",
}
CKPT = {"best": "stratified_eval.json", "last": "stratified_eval_ckpt_last.json"}

rows = []
for exp, rd in RUNS.items():
    for ck, fname in CKPT.items():
        p = rd / fname
        if not p.exists():
            print(f"[miss] {exp}/{ck}")
            continue
        d = json.loads(p.read_text(encoding="utf-8"))
        for split in ("val", "test"):
            strata = d["results"].get(split, {})
            for st in ("event", "quiet", "overall"):
                m = strata.get(st, {})
                if not m or m.get("n", 0) == 0:
                    continue
                rows.append({
                    "exp": exp, "ckpt": ck, "split": split, "stratum": st,
                    "n": m.get("n"),
                    "ccc": m.get("ccc"),
                    "ecm": m.get("event_corr_mean"),
                    "psr": m.get("pred_std_over_gt_std"),
                    "peak_f1": m.get("peak_f1"),
                })

df = pd.DataFrame(rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda v: f"{v:.4f}" if isinstance(v, float) else str(v))

print("========== TEST split: CCC per stratum (决策表) ==========")
t = df[df["split"] == "test"]
pvt = t.pivot_table(index=["exp","ckpt"], columns="stratum", values="ccc")
pvt["event/quiet_ratio"] = pvt["event"] / pvt["quiet"].replace(0, float("nan"))
pvt["event/overall_ratio"] = pvt["event"] / pvt["overall"].replace(0, float("nan"))
print(pvt.round(4))

print("\n========== TEST pred_std/gt_std per stratum ==========")
pvt_psr = t.pivot_table(index=["exp","ckpt"], columns="stratum", values="psr")
print(pvt_psr.round(4))

print("\n========== 决策判断 ==========")
for (exp, ckpt), row in pvt.iterrows():
    ev = row.get("event")
    qt = row.get("quiet")
    ov = row.get("overall")
    if pd.isna(ev) or pd.isna(qt) or pd.isna(ov):
        print(f"{exp} / {ckpt}: 缺数据")
        continue
    ratio = ev / qt if qt != 0 else float("inf")
    notes = []
    if ratio > 2.0:
        notes.append(f"event/quiet={ratio:.2f} > 2 (信号集中在事件区)")
    elif ratio > 1.5:
        notes.append(f"event/quiet={ratio:.2f} (轻度集中)")
    else:
        notes.append(f"event/quiet={ratio:.2f} (信号分散)")
    print(f"{exp:18s} / {ckpt}: CCC event={ev:.4f} quiet={qt:.4f} overall={ov:.4f} — {' | '.join(notes)}")

print("\n========== 跨实验对比（同 horizon post_s=10，不同 out_dim）==========")
for ckpt in ("best", "last"):
    e4 = pvt.loc[("Exp4_post10_d51", ckpt)].get("overall") if ("Exp4_post10_d51", ckpt) in pvt.index else None
    e6 = pvt.loc[("Exp6_post10_d1", ckpt)].get("overall") if ("Exp6_post10_d1", ckpt) in pvt.index else None
    if e4 is not None and e6 is not None and not (pd.isna(e4) or pd.isna(e6)):
        ratio = e6 / e4 if e4 != 0 else float("inf")
        print(f"ckpt={ckpt}: Exp4 (d51)={e4:.4f}  Exp6 (d1)={e6:.4f}  ratio={ratio:.2f}x")
        if ratio > 1.8:
            print("  → output-dim 从 51→1 大幅提升 CCC，支持 seq2seq 方向 (B)")
        elif ratio < 1.3:
            print("  → output-dim 差异不足以解释差距，任务天花板为主 (A)")
        else:
            print("  → output-dim 有贡献但不是唯一瓶颈，A/B 都合理")


========== TEST split: CCC per stratum (决策表) ==========
stratum               event  overall  quiet  event/quiet_ratio  event/overall_ratio
exp             ckpt                                                               
Exp4_post10_d51 best 0.0526   0.0457 0.0388             1.3558               1.1519
                last 0.0447   0.0333 0.0228             1.9636               1.3435
Exp5_post3_d16  best 0.0657   0.0569 0.0473             1.3889               1.1551
                last 0.0793   0.0764 0.0732             1.0823               1.0371
Exp6_post10_d1  best 0.0536   0.0426 0.0299             1.7944               1.2584
                last 0.0976   0.0856 0.0740             1.3185               1.1404

========== TEST pred_std/gt_std per stratum ==========
stratum               event  overall  quiet
exp             ckpt                       
Exp4_post10_d51 best 0.2014   0.2140 0.2261
                last 0.1689   0.1852 0.2001
Exp5_post3_d16  best 0.2063   0.1956 0.1